In [25]:
# Q1: Define required fields for each CSV file
from pathlib import Path
import csv
import re
import json
import math
from datetime import datetime
INPUT_FOLDER = Path(
    r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads"
)
OUTPUT_FOLDER = Path("output")
OUTPUT_FOLDER.mkdir(exist_ok=True)
FILE_SCHEMAS = {
    "olist_customers_dataset.csv": [
        "customer_id", "customer_unique_id", "customer_zip_code_prefix"
    ],
    "olist_geolocation_dataset.csv": [
        "geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"
    ],
    "olist_orders_dataset.csv": [
        "order_id", "customer_id", "order_status", "order_purchase_timestamp"
    ],
    "olist_order_items_dataset.csv": [
        "order_id", "order_item_id", "product_id", "seller_id", "price"
    ],
    "olist_order_payments_dataset.csv": [
        "order_id", "payment_sequential", "payment_type", "payment_value"
    ],
    "olist_order_reviews_dataset.csv": [
        "review_id", "order_id", "review_score"
    ],
    "olist_products_dataset.csv": [
        "product_id"
    ],
    "olist_sellers_dataset.csv": [
        "seller_id", "seller_zip_code_prefix"
    ],
    "product_category_name_translation.csv": [
        "product_category_name", "product_category_name_english"
    ]
}
print("REQUIRED FIELDS FOR OLIST DATASET")
print("-" * 60)
for filename, fields in FILE_SCHEMAS.items():
    print(f"\nFile: {filename}")
    print("Required fields:", fields)


REQUIRED FIELDS FOR OLIST DATASET
------------------------------------------------------------

File: olist_customers_dataset.csv
Required fields: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix']

File: olist_geolocation_dataset.csv
Required fields: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng']

File: olist_orders_dataset.csv
Required fields: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']

File: olist_order_items_dataset.csv
Required fields: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'price']

File: olist_order_payments_dataset.csv
Required fields: ['order_id', 'payment_sequential', 'payment_type', 'payment_value']

File: olist_order_reviews_dataset.csv
Required fields: ['review_id', 'order_id', 'review_score']

File: olist_products_dataset.csv
Required fields: ['product_id']

File: olist_sellers_dataset.csv
Required fields: ['seller_id', 'seller_zip_code_prefix']

File: product_category_name_translation.cs

In [26]:
# Q2: Validate every row and collect errors
def validate_required_fields(row, required_fields):
    errors = []
    for field in required_fields:
        if field not in row:
            errors.append(f"Missing column: {field}")
        elif row[field] is None or row[field].strip() == "":
            errors.append(f"Empty required field: {field}")
    return errors
required_validation_results = {}
for filename, required_fields in FILE_SCHEMAS.items():
    file_path = INPUT_FOLDER / filename
    if not file_path.exists():
        print(f"\nFile not found: {filename}")
        continue
    file_errors = []
    total_rows = 0
    with open(file_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        for row_number, row in enumerate(reader, start=2):
            total_rows += 1
            errors = validate_required_fields(
                row, required_fields
            )
            if errors:
                file_errors.append({
                    "row_number": row_number,
                    "record": row,
                    "reason": "; ".join(errors)
                })
    required_validation_results[filename] = {
        "total_rows": total_rows,
        "rejected_rows": file_errors
    }
    print(f"\nFile: {filename}")
    print("Total rows:", total_rows)
    print("Rejected rows:", len(file_errors))


File: olist_customers_dataset.csv
Total rows: 99441
Rejected rows: 0

File: olist_geolocation_dataset.csv
Total rows: 1000163
Rejected rows: 0

File: olist_orders_dataset.csv
Total rows: 99441
Rejected rows: 0

File: olist_order_items_dataset.csv
Total rows: 112650
Rejected rows: 0

File: olist_order_payments_dataset.csv
Total rows: 103886
Rejected rows: 0

File: olist_order_reviews_dataset.csv
Total rows: 99224
Rejected rows: 0

File: olist_products_dataset.csv
Total rows: 32951
Rejected rows: 0

File: olist_sellers_dataset.csv
Total rows: 3095
Rejected rows: 0

File: product_category_name_translation.csv
Total rows: 71
Rejected rows: 0


In [27]:
# Q3: Save rejected rows into bad_records.csv
def save_bad_records(rejected_records, output_path):
    fieldnames = [
        "source_file", "row_number", "reason", "record_data"
    ]
    with open(
        output_path, "w", encoding="utf-8", newline=""
    ) as file:
        writer = csv.DictWriter(
            file, fieldnames=fieldnames
        )
        writer.writeheader()
        for record in rejected_records:
            writer.writerow({
                "source_file": record["source_file"],
                "row_number": record["row_number"],
                "reason": record["reason"],
                "record_data": json.dumps(
                    record["record"],
                    ensure_ascii=False
                )
            })
rejected_records_q3 = []
for filename, result in required_validation_results.items():
    for rejected in result["rejected_rows"]:
        rejected_records_q3.append({
            "source_file": filename,
            "row_number": rejected["row_number"],
            "reason": rejected["reason"],
            "record": rejected["record"]
        })
bad_records_path = OUTPUT_FOLDER / "bad_records.csv"
save_bad_records(
    rejected_records_q3, bad_records_path
)
print("Rejected records saved successfully.")
print("Output file:", bad_records_path)
print("Total rejected records:", len(rejected_records_q3))

Rejected records saved successfully.
Output file: output\bad_records.csv
Total rejected records: 0


In [28]:
# Q4: Numeric and date validation
NUMERIC_FIELDS = {
    "olist_order_items_dataset.csv": [
        "price", "freight_value"
    ],
    "olist_order_payments_dataset.csv": [
        "payment_sequential", "payment_installments", "payment_value"
    ],
    "olist_order_reviews_dataset.csv": [
        "review_score"
    ],
    "olist_geolocation_dataset.csv": [
        "geolocation_lat", "geolocation_lng"
    ]
}
DATE_FIELDS = {
    "olist_orders_dataset.csv": [
        "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"
    ],
    "olist_order_reviews_dataset.csv": [
        "review_creation_date", "review_answer_timestamp"
    ]
}
def validate_numeric(value, field_name):
    if value is None or value.strip() == "":
        return None
    try:
        number = float(value)
        if not math.isfinite(number):
            return f"{field_name} must be a finite number"
    except ValueError:
        return f"{field_name} must be numeric"
    return None
def validate_date(value, field_name):
    if value is None or value.strip() == "":
        return None
    try:
        datetime.fromisoformat(
            value.strip().replace("Z", "+00:00")
        )
    except ValueError:
        return f"{field_name} must contain a valid date"
    return None
type_validation_results = {}
for filename in FILE_SCHEMAS:
    file_path = INPUT_FOLDER / filename
    if not file_path.exists():
        continue
    errors_list = []
    total_rows = 0
    with open(file_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        for row_number, row in enumerate(reader, start=2):
            total_rows += 1
            row_errors = []
            for field in NUMERIC_FIELDS.get(filename, []):
                error = validate_numeric(
                    row.get(field),
                    field
                )
                if error:
                    row_errors.append(error)
            for field in DATE_FIELDS.get(filename, []):
                error = validate_date(
                    row.get(field),
                    field
                )
                if error:
                    row_errors.append(error)
            if row_errors:
                errors_list.append({
                    "row_number": row_number,
                    "record": row,
                    "reason": "; ".join(row_errors)
                })
    type_validation_results[filename] = {
        "total_rows": total_rows,
        "rejected_rows": errors_list
    }
    print(f"\nFile: {filename}")
    print("Total rows:", total_rows)
    print("Type errors:", len(errors_list))


File: olist_customers_dataset.csv
Total rows: 99441
Type errors: 0

File: olist_geolocation_dataset.csv
Total rows: 1000163
Type errors: 0

File: olist_orders_dataset.csv
Total rows: 99441
Type errors: 0

File: olist_order_items_dataset.csv
Total rows: 112650
Type errors: 0

File: olist_order_payments_dataset.csv
Total rows: 103886
Type errors: 0

File: olist_order_reviews_dataset.csv
Total rows: 99224
Type errors: 0

File: olist_products_dataset.csv
Total rows: 32951
Type errors: 0

File: olist_sellers_dataset.csv
Total rows: 3095
Type errors: 0

File: product_category_name_translation.csv
Total rows: 71
Type errors: 0


In [29]:
# Q5: Business rule validation
ALLOWED_ORDER_STATUS = {
    "created", "approved", "invoiced", "processing", "shipped", "delivered", "unavailable", "canceled"
}
def validate_business_rules(filename, row):
    errors = []
    if filename == "olist_order_items_dataset.csv":
        try:
            price = float(row.get("price", ""))
            if price <= 0:
                errors.append("Price must be greater than zero")
        except (ValueError, TypeError):
            errors.append("Price must be numeric")
    if filename == "olist_orders_dataset.csv":
        order_status = row.get("order_status", "").strip().lower()
        if order_status not in ALLOWED_ORDER_STATUS:
            errors.append(
                f"Invalid order_status: {order_status}"
            )
    return errors
business_validation_results = {}
for filename in FILE_SCHEMAS:
    file_path = INPUT_FOLDER / filename
    if not file_path.exists():
        continue
    errors_list = []
    total_rows = 0
    with open(file_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        for row_number, row in enumerate(reader, start=2):
            total_rows += 1
            errors = validate_business_rules(
                filename,
                row
            )
            if errors:
                errors_list.append({
                    "row_number": row_number,
                    "record": row,
                    "reason": "; ".join(errors)
                })
    business_validation_results[filename] = {
        "total_rows": total_rows,
        "rejected_rows": errors_list
    }
    print(f"\nFile: {filename}")
    print("Total rows:", total_rows)
    print("Business rule errors:", len(errors_list))


File: olist_customers_dataset.csv
Total rows: 99441
Business rule errors: 0

File: olist_geolocation_dataset.csv
Total rows: 1000163
Business rule errors: 0

File: olist_orders_dataset.csv
Total rows: 99441
Business rule errors: 0

File: olist_order_items_dataset.csv
Total rows: 112650
Business rule errors: 0

File: olist_order_payments_dataset.csv
Total rows: 103886
Business rule errors: 0

File: olist_order_reviews_dataset.csv
Total rows: 99224
Business rule errors: 0

File: olist_products_dataset.csv
Total rows: 32951
Business rule errors: 0

File: olist_sellers_dataset.csv
Total rows: 3095
Business rule errors: 0

File: product_category_name_translation.csv
Total rows: 71
Business rule errors: 0


In [30]:
# Q6: Regex validation
ID_PATTERN = re.compile(r"^[0-9a-fA-F]{32}$")
ZIP_PATTERN = re.compile(r"^\d{5}$")
def validate_regex_fields(filename, row):
    errors = []
    for field, value in row.items():
        if value is None or value.strip() == "":
            continue
        value = value.strip()
        if field.endswith("_id") and field != "order_item_id":
            if not ID_PATTERN.fullmatch(value):
                errors.append(
                    f"{field} must be a 32-character hexadecimal ID"
                )
        if field.endswith("_zip_code_prefix"):
            if not ZIP_PATTERN.fullmatch(value):
                errors.append(
                    f"{field} must contain exactly 5 digits"
                )
    return errors
regex_validation_results = {}
for filename in FILE_SCHEMAS:
    file_path = INPUT_FOLDER / filename
    if not file_path.exists():
        continue
    errors_list = []
    total_rows = 0
    with open(file_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        for row_number, row in enumerate(reader, start=2):
            total_rows += 1
            errors = validate_regex_fields(
                filename, row
            )
            if errors:
                errors_list.append({
                    "row_number": row_number,
                    "record": row,
                    "reason": "; ".join(errors)
                })
    regex_validation_results[filename] = {
        "total_rows": total_rows,
        "rejected_rows": errors_list
    }
    print(f"\nFile: {filename}")
    print("Total rows:", total_rows)
    print("Regex errors:", len(errors_list))


File: olist_customers_dataset.csv
Total rows: 99441
Regex errors: 0

File: olist_geolocation_dataset.csv
Total rows: 1000163
Regex errors: 0

File: olist_orders_dataset.csv
Total rows: 99441
Regex errors: 0

File: olist_order_items_dataset.csv
Total rows: 112650
Regex errors: 0

File: olist_order_payments_dataset.csv
Total rows: 103886
Regex errors: 0

File: olist_order_reviews_dataset.csv
Total rows: 99224
Regex errors: 0

File: olist_products_dataset.csv
Total rows: 32951
Regex errors: 0

File: olist_sellers_dataset.csv
Total rows: 3095
Regex errors: 0

File: product_category_name_translation.csv
Total rows: 71
Regex errors: 0


In [31]:
# Q7: Custom exceptions
class SchemaError(Exception):
    """Raised when required columns are missing."""
    pass
class FileFormatError(Exception):
    """Raised when the CSV file has an invalid format."""
    pass
def check_file_schema(file_path, required_fields):
    try:
        with open(
            file_path, "r", encoding="utf-8-sig", newline=""
        ) as file:
            reader = csv.DictReader(file)
            if reader.fieldnames is None:
                raise FileFormatError(
                    f"CSV file has no header: {file_path.name}"
                )
            if len(reader.fieldnames) != len(set(reader.fieldnames)):
                raise FileFormatError(
                    f"Duplicate column names in {file_path.name}"
                )
            missing_fields = [
                field for field in required_fields
                if field not in reader.fieldnames
            ]
            if missing_fields:
                raise SchemaError(
                    f"Missing columns in {file_path.name}: {missing_fields}"
                )
    except UnicodeDecodeError as error:
        raise FileFormatError(
            f"Unable to decode file: {file_path.name}"
        ) from error
print("Custom exceptions created successfully.")
print("SchemaError and FileFormatError are ready to use.")

Custom exceptions created successfully.
SchemaError and FileFormatError are ready to use.


In [32]:
# Q8: Handle missing and empty files
def read_csv_safely(file_path, required_fields):
    if not file_path.exists():
        print(f"File is missing: {file_path.name}")
        return None
    if file_path.stat().st_size == 0:
        print(f"File is empty: {file_path.name}")
        return None
    try:
        check_file_schema(
            file_path, required_fields
        )
        with open(
            file_path, "r", encoding="utf-8-sig", newline=""
        ) as file:
            reader = csv.DictReader(file)
            rows = []
            for row in reader:
                if None in row:
                    raise FileFormatError(
                        f"Extra values found in a row of {file_path.name}"
                    )
                rows.append(row)
            return rows
    except (SchemaError, FileFormatError) as error:
        print(f"File validation error: {error}")
        return None
    except UnicodeDecodeError:
        print(f"Invalid file encoding: {file_path.name}")
        return None
safe_file_results = {}
for filename, required_fields in FILE_SCHEMAS.items():
    file_path = INPUT_FOLDER / filename
    rows = read_csv_safely(
        file_path, required_fields
    )
    safe_file_results[filename] = rows
    if rows is not None:
        print(f"{filename}: {len(rows)} rows read successfully.")

olist_customers_dataset.csv: 99441 rows read successfully.
olist_geolocation_dataset.csv: 1000163 rows read successfully.
olist_orders_dataset.csv: 99441 rows read successfully.
olist_order_items_dataset.csv: 112650 rows read successfully.
olist_order_payments_dataset.csv: 103886 rows read successfully.
olist_order_reviews_dataset.csv: 99224 rows read successfully.
olist_products_dataset.csv: 32951 rows read successfully.
olist_sellers_dataset.csv: 3095 rows read successfully.
product_category_name_translation.csv: 71 rows read successfully.


In [ ]:
# Q9: Print validation summary
def validate_complete_row(filename, row, required_fields):
    errors = []
    errors.extend(
        validate_required_fields(
            row, required_fields
        )
    )
    for field in NUMERIC_FIELDS.get(filename, []):
        error = validate_numeric(
            row.get(field), field
        )
        if error:
            errors.append(error)
    for field in DATE_FIELDS.get(filename, []):
        error = validate_date(
            row.get(field), field
        )
        if error:
            errors.append(error)
    errors.extend(
        validate_business_rules(
            filename, row
        )
    )
    errors.extend(
        validate_regex_fields(
            filename, row
        )
    )
    return list(dict.fromkeys(errors))
validation_summary = {}
for filename, rows in safe_file_results.items():
    if rows is None:
        continue
    total_rows = len(rows)
    valid_rows = 0
    rejected_rows = 0
    for row in rows:
        errors = validate_complete_row(
            filename, row, FILE_SCHEMAS[filename]
        )
        if errors:
            rejected_rows += 1
        else:
            valid_rows += 1
    validation_summary[filename] = {
        "total_rows": total_rows,
        "valid_rows": valid_rows,
        "rejected_rows": rejected_rows
    }
print("\nOLIST DATA VALIDATION SUMMARY")
print("=" * 75)
for filename, summary in validation_summary.items():
    print(f"\nFile: {filename}")
    print("Total rows:", summary["total_rows"])
    print("Valid rows:", summary["valid_rows"])
    print("Rejected rows:", summary["rejected_rows"])
print("\nOVERALL SUMMARY")
print("=" * 75)
total = sum(
    result["total_rows"]
    for result in validation_summary.values()
)
valid = sum(
    result["valid_rows"]
    for result in validation_summary.values()
)
rejected = sum(
    result["rejected_rows"]
    for result in validation_summary.values()
)
print("Total rows:", total)
print("Valid rows:", valid)
print("Rejected rows:", rejected)


========== VALIDATION SUMMARY ==========
Total rows    : 1000
Valid rows    : 950
Rejected rows : 50
Rejection percentage: 5.00%


In [34]:
def validate_complete_row(filename, row, required_fields):

    errors = []

    # 1. Required field validation
    errors.extend(
        validate_required_fields(row, required_fields)
    )

    # 2. Numeric validation
    for field in NUMERIC_FIELDS.get(filename, []):

        error = validate_numeric(
            row.get(field),
            field
        )

        if error:
            errors.append(error)

    # 3. Date validation
    for field in DATE_FIELDS.get(filename, []):

        error = validate_date(
            row.get(field),
            field
        )

        if error:
            errors.append(error)

    # 4. Business rule validation
    errors.extend(
        validate_business_rules(filename, row)
    )

    # 5. Regex validation
    errors.extend(
        validate_regex_fields(filename, row)
    )

    # Remove duplicate error messages
    return list(dict.fromkeys(errors))


print("validate_complete_row() defined successfully.")

validate_complete_row() defined successfully.


In [35]:
MAX_REJECTION_PERCENTAGE = 20
final_summary = []
all_rejected_records = []
for filename, required_fields in FILE_SCHEMAS.items():
    file_path = INPUT_FOLDER / filename
    print(f"\nProcessing: {filename}")
    print("-" * 60)
    rows = read_csv_safely(
        file_path,
        required_fields
    )
    if rows is None:
        final_summary.append({
            "file_name": filename,
            "total_rows": 0,
            "valid_rows": 0,
            "rejected_rows": 0,
            "rejection_percentage": 0,
            "status": "SKIPPED"
        })
        continue
    total_rows = len(rows)
    valid_rows = 0
    rejected_rows = 0
    for row_number, row in enumerate(rows, start=2):
        errors = validate_complete_row(
            filename, row, required_fields
        )
        if errors:
            rejected_rows += 1
            all_rejected_records.append({
                "source_file": filename,
                "row_number": row_number,
                "reason": "; ".join(errors),
                "record": row
            })
        else:
            valid_rows += 1
    if total_rows > 0:
        rejection_percentage = (rejected_rows / total_rows) * 100
    else:
        rejection_percentage = 0
    if rejection_percentage > MAX_REJECTION_PERCENTAGE:
        status = "FAILED"
    else:
        status = "PASSED"
    final_summary.append({
        "file_name": filename,
        "total_rows": total_rows,
        "valid_rows": valid_rows,
        "rejected_rows": rejected_rows,
        "rejection_percentage": round(rejection_percentage,2),
        "status": status
    })
    print("Total rows:", total_rows)
    print("Valid rows:", valid_rows)
    print("Rejected rows:", rejected_rows)
    print("Rejection percentage:", round(rejection_percentage, 2), "%")
    print("File status:", status)
# Save all rejected records into one CSV file
save_bad_records(
    all_rejected_records,
    OUTPUT_FOLDER / "bad_records.csv"
)
# Save final summary into a CSV file
summary_path = OUTPUT_FOLDER / "validation_summary.csv"
with open(
    summary_path, "w", encoding="utf-8", newline=""
) as file:
    fieldnames = [
        "file_name", "total_rows", "valid_rows", "rejected_rows", "rejection_percentage", "status"
    ]
    writer = csv.DictWriter(
        file, fieldnames=fieldnames
    )
    writer.writeheader()
    writer.writerows(final_summary)
print("\nFINAL VALIDATION REPORT")
print("=" * 75)
for result in final_summary:
    print(result)
print("\nRejected records saved to:", OUTPUT_FOLDER / "bad_records.csv")
print("Validation summary saved to:", summary_path)


Processing: olist_customers_dataset.csv
------------------------------------------------------------
Total rows: 99441
Valid rows: 99441
Rejected rows: 0
Rejection percentage: 0.0 %
File status: PASSED

Processing: olist_geolocation_dataset.csv
------------------------------------------------------------
Total rows: 1000163
Valid rows: 1000163
Rejected rows: 0
Rejection percentage: 0.0 %
File status: PASSED

Processing: olist_orders_dataset.csv
------------------------------------------------------------
Total rows: 99441
Valid rows: 99441
Rejected rows: 0
Rejection percentage: 0.0 %
File status: PASSED

Processing: olist_order_items_dataset.csv
------------------------------------------------------------
Total rows: 112650
Valid rows: 112650
Rejected rows: 0
Rejection percentage: 0.0 %
File status: PASSED

Processing: olist_order_payments_dataset.csv
------------------------------------------------------------
Total rows: 103886
Valid rows: 103886
Rejected rows: 0
Rejection percentag